In [1]:
! pip install sk-learn 

ERROR: Could not find a version that satisfies the requirement sk-learn (from versions: none)
ERROR: No matching distribution found for sk-learn


In [5]:
!pip install joblib

! pip -m install sk-learn 





[optparse.groups]Usage:[/]   
  pip <command> \[options]

no such option: -m


In [2]:
import joblib
scaler = joblib.load("scaler_v2_balanced.save")
print("Scaler expects features:", scaler.n_features_in_)


Scaler expects features: 5


c:\Users\Abhishek\Desktop\project\ann\ann rain predition\.venv\lib\site-packages\sklearn\base.py:348: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.6.1 when using version 1.3.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [ ]:
import streamlit as st
import pandas as pd
import numpy as np
import tensorflow as tf
import joblib
import requests
from geopy.geocoders import Nominatim
import plotly.express as px

# -------------------------------------------------
# PAGE CONFIG
# -------------------------------------------------
st.set_page_config(
    page_title="Rain AI",
    page_icon="🌧️",
    layout="centered"
)

# -------------------------------------------------
# MODERN UI STYLE
# -------------------------------------------------
st.markdown("""
<style>
body {
    background-color:#0f172a;
}

.block-container {
    padding-top: 2rem;
}

/* Hero Card */
.hero {
    background: linear-gradient(135deg,#1e293b,#0f172a);
    padding:40px;
    border-radius:25px;
    text-align:center;
    color:white;
    margin-bottom:25px;
    box-shadow:0 10px 40px rgba(0,0,0,0.5);
}

.temp {
    font-size:80px;
    font-weight:700;
    margin:0;
}

.city {
    font-size:20px;
    opacity:0.7;
}

/* Metric cards */
.metric-card {
    background:#111827;
    padding:20px;
    border-radius:20px;
    text-align:center;
    box-shadow:0 4px 15px rgba(0,0,0,0.4);
}
.metric-value {
    font-size:26px;
    font-weight:600;
    color:white;
}
.metric-label {
    font-size:12px;
    opacity:0.6;
}
</style>
""", unsafe_allow_html=True)


st.title("🌧️ Rainfall Prediction AI")

# -------------------------------------------------
# LOAD MODEL
# -------------------------------------------------
@st.cache_resource
def load_assets():
    model = tf.keras.models.load_model("rainfall_model_v2_balanced.h5")
    scaler = joblib.load("scaler_v2_balanced.save")
    return model, scaler

model, scaler = load_assets()


# -------------------------------------------------
# FETCH WEATHER
# -------------------------------------------------
def fetch_history(lat, lon):

    url = "https://api.open-meteo.com/v1/forecast"

    params = {
        "latitude": lat,
        "longitude": lon,
        "hourly": "temperature_2m,relative_humidity_2m,wind_speed_10m,surface_pressure,shortwave_radiation",
        "current": "temperature_2m,relative_humidity_2m,wind_speed_10m,surface_pressure",
        "past_days": 14,
        "timezone": "auto"
    }

    r = requests.get(url, params=params).json()

    df = pd.DataFrame(r["hourly"])
    df["time"] = pd.to_datetime(df["time"])

    hist = df.resample("D", on="time").agg({
        "temperature_2m": "mean",
        "relative_humidity_2m": "mean",
        "wind_speed_10m": "max",
        "surface_pressure": "mean",
        "shortwave_radiation": "sum"
    }).tail(14)

    hist["surface_pressure"] /= 10
    hist["shortwave_radiation"] /= 1000

    return hist, r["current"]


# -------------------------------------------------
# INPUT BAR
# -------------------------------------------------
with st.form("predict_form", clear_on_submit=False):

    city = st.text_input("City name", "Mumbai")

    run = st.form_submit_button(
        "🌧 Predict Rain",
        use_container_width=True
    )



# -------------------------------------------------
# PREDICTION
# -------------------------------------------------
if run:

    geo = Nominatim(user_agent="rain_ai").geocode(city)

    if not geo:
        st.error("City not found")
        st.stop()

    lat, lon = geo.latitude, geo.longitude

    with st.spinner("Analyzing weather..."):
        hist, current = fetch_history(lat, lon)

    # ----- rename for scaler -----
    model_df = hist.copy()
    model_df.columns = ["temp","humidity","wind","pressure","solar"]

    X = scaler.transform(model_df).reshape(1,14,5)

    prob = float(model.predict(X)[0][0]) * 100

    # -------------------------------------------------
    # HERO CARD
    # -------------------------------------------------
    st.markdown(f"""
        <div class="hero">
            <p class="temp">{current['temperature_2m']}°C</p>
            <p class="city">{city.title()}</p>
            <h3>🌧 Rain Probability: {prob:.1f}%</h3>
        </div>
    """, unsafe_allow_html=True)

    # -------------------------------------------------
    # METRICS
    # -------------------------------------------------
    c1, c2, c3 = st.columns(3)

    metrics = [
        ("Humidity", f"{current['relative_humidity_2m']}%"),
        ("Wind", f"{current['wind_speed_10m']} km/h"),
        ("Pressure", f"{current['surface_pressure']:.0f} mb")
    ]

    for col, (label, value) in zip([c1, c2, c3], metrics):
        col.markdown(f"""
            <div class="metric-card">
                <div class="metric-value">{value}</div>
                <div class="metric-label">{label}</div>
            </div>
        """, unsafe_allow_html=True)

    # -------------------------------------------------
    # MINI TREND CHART
    # -------------------------------------------------
    st.markdown("<br>", unsafe_allow_html=True)

    trend = hist.copy()
    trend["temp"] = trend["temperature_2m"]

    fig = px.line(trend, y="temp", title="14-Day Temperature Trend")
    fig.update_layout(
        paper_bgcolor="#0f172a",
        plot_bgcolor="#0f172a",
        font_color="white"
    )

    st.plotly_chart(fig, use_container_width=True)
